

```
🟦 DAY 20 — LANGCHAIN (BASIC TO ADVANCED)

1. GENAI — 60 MIN
□ LangChain Core Architecture (LLMs, Prompts, and Output Parsers)
□ Build structured chains using LangChain Expression Language (LCEL)
□ Implement Memory (ConversationBufferMemory, WindowMemory) to add state
□ Integrate external Tools and build intelligent Agents
□ Setup RAG basics (Document Loaders, Text Splitters, and VectorStore integration)
```



##### **LANGCHAIN**

> Langchain is a Framework for developers application to work with LLMs

> Link : https://smith.langchain.com/o/d5e687d9-1f08-425d-853e-0aa8cbc1fde7

## Section 0: Environment Setup & Package Installation

Why it is used: This section initializes the computational environment, ensuring all necessary orchestration libraries and third-party tools are present. It further configures secure access to remote inference endpoints.

Key Parameters & Why They Are Used:
- `langchain`: The core framework for LLM orchestration.
- `langchain-nvidia-ai-endpoints`: Integration for high-performance NVIDIA hardware acceleration.
- `pypdf`: Required for parsing unstructured PDF binary data.

Logic/Architecture Diagram:
```text
[Colab Runtime] ◆ [PyPI Package Manager] ◆ [OS Environment Variables] ◆ [Ready State]
```

Code Implementation:

In [46]:
!pip install -q langchain langchain-community langchain-nvidia-ai-endpoints langchain-huggingface openai wikipedia numexpr pypdf

#### **02 : Set up Environment**

In [54]:
import os

# Configure secure API access for NVIDIA and HuggingFace services
os.environ["OPENAI_API_KEY"] = "API_KEY"
os.environ["HUGGINGFACE_API_TOKEN"] = "API_KEY"

## Section 1: Large Language Model (LLM) Wrappers

Why it is used: LLM wrappers provide a standardized interface to interact with various model providers, abstracting the underlying API calls into a unified `invoke` method.

Key Parameters & Why They Are Used:
- `model`: Defines the specific neural architecture (e.g., gpt-oss-20b) for inference.
- `temperature`: Set to 0.5 to balance deterministic logic with natural language fluidness.
- `max_completion_tokens`: Constrains the output length to optimize latency.

Logic/Architecture Diagram:
```text
[Input Query] ❖ [ChatNVIDIA Wrapper] ❖ [Remote API Endpoint] ❖ [AI Response]
```

Code Implementation:

### **NVIDIA API**

##### **EXAMPLE 1**

In [5]:
!pip install openai

In [6]:
!pip install langchain-nvidia-ai-endpoints

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.8 MB/s eta 0:00:00


In [48]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
    model="openai/gpt-oss-20b",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.5,
    max_completion_tokens=1024
)

response = llm.invoke("Which number is larger, 9.11 or 9.8?")
print(response.content)

9.8 is larger.

**Why?**  
- 9.11 means 9 + 0.11 (which is 9.11).  
- 9.8 means 9 + 0.8 (which is 9.80).  

Since 0.8 > 0.11, the whole number 9.8 is greater than 9.11.


#### **HUGGINNG FACE MODEL**

##### **EXAMPLE 1**

In [8]:
!pip install huggingface_hub

In [9]:
!pip install -U langchain-huggingface

In [10]:
from langchain_community.llms import HuggingFaceHub

/tmp/ipykernel_847/2935558610.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFaceHub


In [11]:
"""
from langchain_huggingface import HuggingFaceEndpoint
import os

Llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta", # Switched to a highly available model
    task="conversational",                 # Updated task for the new model
    huggingfacehub_api_token=os.environ["HUGGINGFACE_API_TOKEN"],
    temperature=0.7,
    max_new_tokens=64
)

result = Llm.invoke("Which number is larger, 9.11 or 9.8?")
print(result)
"""

'\nfrom langchain_huggingface import HuggingFaceEndpoint\nimport os\n\nLlm = HuggingFaceEndpoint(\n    repo_id="HuggingFaceH4/zephyr-7b-beta", # Switched to a highly available model\n    task="conversational",                 # Updated task for the new model\n    huggingfacehub_api_token=os.environ["HUGGINGFACE_API_TOKEN"],\n    temperature=0.7,\n    max_new_tokens=64\n)\n\nresult = Llm.invoke("Which number is larger, 9.11 or 9.8?")\nprint(result)\n'

##### **EXAMPLE 2**

In [12]:
"""
from langchain import HuggingFaceHub

repo_id = "google/flan-t5-xxl"  # See https://huggingface.co/models?pipeline_tag=text-generation&sort=downloads for some other options

llm = HuggingFaceHub(repo=repo_id, model_kwargs={"temperature":0, "max_length":64})

name = llm.predict("Which number is larger, 9.11 or 9.8?")
print(name)
"""

'\nfrom langchain import HuggingFaceHub\n\nrepo_id = "google/flan-t5-xxl"  # See https://huggingface.co/models?pipeline_tag=text-generation&sort=downloads for some other options\n\nllm = HuggingFaceHub(repo=repo_id, model_kwargs={"temperature":0, "max_length":64})\n\nname = llm.predict("Which number is larger, 9.11 or 9.8?")\nprint(name)\n'

## Section 2: Prompt Templates & LCEL Composition

Why it is used: Prompt Templates formalize the interaction between user data and model instructions. LangChain Expression Language (LCEL) uses the pipe operator to compose these into a single execution unit.

Key Parameters & Why They Are Used:
- `template`: The structured instruction string with dynamic placeholders.
- `input_variables`: Explicitly declares the keys required for the formatting engine.

Logic/Architecture Diagram:
```text
[User Variable] ◆ [PromptTemplate] ◆ [| Pipe Operator] ◆ [LLM Object]
```

Code Implementation:

##### **EXAMPLE 1**

In [13]:
from langchain_core.prompts import PromptTemplate

#CREATE A PROMPT TEMPLATE
prompt_template_name = PromptTemplate(
    input_variables = ['cusine'],
    template = "I want to open restraunt for {cusine} food. Suggest a fency name for this"
)

#USE .fromat() to give value
p = prompt_template_name.format(cusine = "Italian")
print(p)

I want to open restraunt for Italian food. Suggest a fency name for this


##### **EXAMPLE 2**

In [14]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("What is a good name for company that makes {products}")
prompt.format(products="colorful socks")


'What is a good name for company that makes colorful socks'

# 🔗 Section 4: LangChain Expression Language (LCEL)

### Why it is used
LCEL is a declarative way to compose chains. It uses the `|` (pipe) operator to stream data between components efficiently and with built-in support for parallel execution.

### Parameters Used & Why
- `chain = prompt | llm`: The pipe syntax replaces legacy `LLMChain` calls for cleaner, more maintainable code.

### ASCII Chain Diagram
```text
   Prompt              LLM             Parser
[ Template ] | [ ChatNVIDIA ] | [ StrOutputParser ]
```

### Conclusion
LCEL makes complex multi-step AI workflows readable and easy to debug.

In [15]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key=os.environ["OPENAI_API_KEY"],
  temperature=1,
  top_p=1,
  max_tokens=2086,
)

/tmp/ipykernel_847/1839722607.py:3: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  llm = ChatNVIDIA(


In [16]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
prompt.format(product="colorful socks")

'What is a good name for a company that makes colorful socks'

In [49]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("What is a good name for a company that makes {product}")
chain = prompt | llm

response = chain.invoke({"product": "colorful socks"})
print(response.content)

Below are a handful of brand‑name ideas that instantly convey the fun, bright, and “sock‑centric” vibe you’re looking for.  For each name I’ve added a quick tagline and a one‑sentence note on why it works (e.g., wordplay, visual imagery, easy to spell, etc.).  Feel free to mix and match or tweak any of them to fit your exact vision.

| # | Company Name | Tagline | Why It Works |
|---|---------------|---------|--------------|
| 1 | **Sock Spectrum** | “Step into every color.” | Spectrum evokes a full range of hues; “Sock” keeps it clear. Short, easy to remember. |
| 2 | **Vivid Toes** | “Brighten your stride.” | “Vivid” signals boldness; “Toes” is playful and directly references socks. |
| 3 | **HueSock** | “Color your feet.” | Combines “hue” (color) with “sock” for a snappy, brand‑able word. |
| 4 | **Rainbow Runners** | “Run with color.” | “Rainbow” is instantly visual; “Runners” hints at athletic socks, but works for casual. |
| 5 | **KaleidoKicks** | “Every step a masterpiece.” | “K

### **BASICALLY THE PROMPT TEMPLATE TAKES OF 5 STEPS**

1. CREATE A MODEL
2. CREATE A PROMPT TEMPLATE --> **{input_variable and template}**
3. FORMAT THE VARIABLE
4. CHIAN USING | THE PROMPT TEMPLATE FIRST THEN LLM
5. THEN INVOKE

<br>


###### **Can we combine multiple PromptTemplates, we will try to combine multiple PromptTemplate**

**the output from the first PromptTemplate is passed to the next PromptTemplate**

#### **To Combine the chain and to set a sequnece for that we use Simple Sequential Chain**

### **Simple Sequential Chain**

In [18]:
# CRETA A MODEL
llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key=os.environ["OPENAI_API_KEY"],
  temperature=1,
  top_p=1,
  max_completion_tokens=2086, # Updated from max_tokens
  timeout=120                 # Increased timeout to handle slow responses
)

#CREATE A PROMPT TEMPLATE
prompt_template_name = PromptTemplate(
    input_variables = ['cusine'],
    template = "I want to open restraunt for {cusine} food. Suggest a fency name for this"
)

#CHAIN THEM TOGEATHER
name_chain = prompt_template_name | llm

#IF NEEDED ANOTHER PROMPT TEMPLATE
prompt_template_items = PromptTemplate(
    input_variables = ['restraunt_name'],
    template = "Suggest some menu items for {restraunt_name}"
)

#ANOTHER CHAIN
food_items_chain = prompt_template_items | llm

In [19]:
# In modern LangChain (LCEL), we can simply pipe chains together.
# The output of name_chain will be passed as input to food_items_chain.
# We use a lambda to map the output string of the first chain to the dictionary key expected by the second.

chain = name_chain | (lambda x: {"restraunt_name": x.content}) | food_items_chain

# Executing the sequential chain
response = chain.invoke({"cusine": "Italian"})
print(response.content)

Below is a **starter‑to‑dessert plate‑form** of menu ideas that fits the 10 restaurant concepts you outlined.  
For each venue I keep the flavor profile true to Italy, but the dish names are a touch “cheesy” (playful, pun‑laden) while staying polished enough for fine‑dining or contemporary‑trattoria settings.  
Feel free to cherry‑pick dishes, swap recipes, or tweak any name that doesn’t sit with you.

| # | Restaurant | A “Cheesy” Item‑Level Menu (Antipasto – Primo – Secondo – Dolce – Drink) |
|---|------------|-----------------------------------------------------------------------|
| 1 | **La Casa dei Sapori** | **Antipasto** – *“Minutini Mosaico”*<br>Vintage‑style tomato‑pilaf with burrata foam and a drizzle of basil‑infused olive oil.<br> **Primo** – *“Cappellini al Caldo”* – classic ragù on owl‑wing butter‑rich gnocchi.<br> **Secondo** – *“Carne di Casa”* – wood‑smoked pork chop with rosemary jus, paired with roasted polenta.<br> **Dolce** – *“Limoncello, Biscotti, L’amico”* – lem

## Section 3: Intelligent Agents & External Tools

Why it is used: Agents expand LLM capabilities by allowing the model to choose and execute external tools like Wikipedia or mathematical engines to verify facts or perform calculations.

Key Parameters & Why They Are Used:
- `tools`: The collection of functional utilities available to the agent.
- `agent`: The specific reasoning strategy (ReAct) used for decision-making.
- `handle_parsing_errors`: Set to True to recover from malformed model outputs.

Logic/Architecture Diagram:
```text
[Input] ❖ [Thought] ❖ [Tool Selection] ❖ [Observation] ❖ [Final Answer]
```

Code Implementation:

**Agent involve of LLM making decisions about which Actions to take taking that action , seeing the Observation and repeating it until it Done**

**When used correctly agents can be extremely powerfull  in order to load the agents One should understand the following concepts**

**1. Tool : A function that performs specific duty. This can be things like : Google Search, Database Lookup, Python Repel, other Chains**

**2. LLM : Large Language Model Powering the agent**

**3. Agent : Agent to use**


> Agent is a very powerfull Concept in the Langchain


> **for example i travel form Dubai to Canada I type this to ChatGPT**

-> Give me Two options from Dubai to Canada on September 1, 2024 | ChatGPT will not be able to answer because has Knoweledge till Sepetember 2021

ChatGPT plus has Expedia Plugins if we enable this plugin it will  got to the Expedia Plugin and will try to pull information about the Flight and pull out the information


> SerpApi is a real time api for Google Search results

##### **Wikipedia and llm-math-tool**

In [20]:
!pip install wikipedia numexpr mypy_extensions langchain

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=48f60ba6c42abcc4b1d86a75ebe69d4ccda9bd4a2f33347555f54d5bcffc52ad
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia




```
This code initializes a LangChain Agent using the ChatNVIDIA large language model.

1.Initialization: It creates an llm instance with a temperature of 0 for deterministic answers.

2.Tool Loading: It loads two external tools—wikipedia for searching information and llm-math for performing calculations.

3.Agent Setup: It initializes a CHAT_ZERO_SHOT_REACT_DESCRIPTION agent. This type of agent follows a 'Thought-Action-Observation' loop, meaning it thinks about the user's question, decides which tool to use, looks at the result, and repeats until it has the final answer.

4.Execution: The agent.invoke command asks the agent to solve a math problem ('What is 25 divided by 5?'). As seen in the output, the agent correctly identified the result.

```



In [50]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_classic.agents import initialize_agent, AgentType

tools = load_tools(["wikipedia", "llm-math"], llm=llm)

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

response = agent.invoke("What is 25 divided by 5?")
print(response["output"])



> Entering new AgentExecutor chain...
Question: What is 25 divided by 5?  
Thought: 25 ÷ 5 equals 5.  
Final Answer: 5

> Finished chain.
5


## Section 4: Conversational Memory Systems

Why it is used: Memory components enable the persistence of state across multiple turns, allowing the LLM to reference earlier parts of the dialogue.

Key Parameters & Why They Are Used:
- `MessagesPlaceholder`: Allocates space in the prompt for injected history.
- `session_id`: Keys the conversation history to a specific user session.

Logic/Architecture Diagram:
```text
[Session Store] ◆ [History Retrieval] ◆ [Prompt Generation] ◆ [State Update]
```

Code Implementation:



```
This code sets up a text generation component using LangChain.

1.LLM Initialization: It creates an instance of ChatNVIDIA using a specific model (gpt-oss-20b). Parameters like temperature (0.5 for a balance of creativity and focus) and max_completion_tokens are set to control the model's output.

2.Prompt Template: It defines a PromptTemplate. This acts as a reusable blueprint for prompts. Instead of writing the full request every time, you use the {cusine} placeholder to dynamically insert different food types into the instruction: 'I want a good name for a restaurant that serves {cusine} food. Suggest a fancy name for this.'

```



In [22]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import PromptTemplate

# Increasing max_completion_tokens to avoid truncated output
llm = ChatNVIDIA(
    model = "openai/gpt-oss-20b",
    api_key = os.environ["OPENAI_API_KEY"],
    temperature = 0.5,
    timeout = 120,
    max_completion_tokens = 1024
)

prompt_template = PromptTemplate(
    input_variables=['cusine'],
    template = "I want a good name for a restraunt that serves {cusine} food. Suggest a fency name for this"
)



```
This code demonstrates the final steps of creating and executing a LangChain pipeline:

1.Chain Construction: The line name_temp = prompt_template | llm uses the LangChain Expression Language (LCEL). The pipe operator (|) connects the prompt template to the model, meaning the output of the prompt is automatically sent as input to the model.

2.Invoking the Chain: The invoke method triggers the process. By passing {"cusine":"Indian"}, LangChain fills the {cusine} placeholder in the template and sends the finished prompt to the NVIDIA AI endpoint.

3.Output Handling: Large Language Models return complex objects containing metadata (like token usage). Using .content extracts just the text response, which is then printed to the console.
```



In [23]:
name_temp =  prompt_template | llm

# By using .content, we display only the text and not the metadata
response = name_temp.invoke({"cusine":"Indian"}) #Can change the cusine to italian , german ..etc to get different output
print(response.content)

Here are a handful of upscale‑savvy names that evoke the richness, aroma, and heritage of Indian cuisine—each paired with a quick tagline to help you picture the vibe:

| # | Restaurant Name | Tagline / Concept | Why It Works |
|---|-------------------|-------------------|--------------|
| 1 | **Saffron Silk** | “Where every dish is a fragrant tapestry.” | Saffron is a luxury spice; “silk” hints at smooth, indulgent flavors. |
| 2 | **Curry Couture** | “Haute cuisine, haute taste.” | “Couture” signals high fashion; pairs perfectly with the artistry of curry. |
| 3 | **Taj Mahal Bistro** | “An edible masterpiece.” | The Taj Mahal is synonymous with opulence and romance—great for a fine‑dining experience. |
| 4 | **Spice Symphony** | “A melodic journey from the heart of India.” | “Symphony” evokes harmony and complexity, like a well‑balanced spice blend. |
| 5 | **Madhur Mingle** | “Sweet conversations, savory stories.” | “Madhur” means sweet in Hindi; the name invites a social, intimate

In [24]:
# The 'response' object is an AIMessage, which holds the text in '.content'
# It does not have a '.memory' attribute.
print("Response Content:", response.content)

# Note: In LCEL (the '|' syntax), memory is not stored in the response.
# To implement memory, you would use 'RunnableWithMessageHistory'.
# For now, you can see the metadata of the response like this:
print("Metadata:", response.response_metadata)

Response Content: Here are a handful of upscale‑savvy names that evoke the richness, aroma, and heritage of Indian cuisine—each paired with a quick tagline to help you picture the vibe:

| # | Restaurant Name | Tagline / Concept | Why It Works |
|---|-------------------|-------------------|--------------|
| 1 | **Saffron Silk** | “Where every dish is a fragrant tapestry.” | Saffron is a luxury spice; “silk” hints at smooth, indulgent flavors. |
| 2 | **Curry Couture** | “Haute cuisine, haute taste.” | “Couture” signals high fashion; pairs perfectly with the artistry of curry. |
| 3 | **Taj Mahal Bistro** | “An edible masterpiece.” | The Taj Mahal is synonymous with opulence and romance—great for a fine‑dining experience. |
| 4 | **Spice Symphony** | “A melodic journey from the heart of India.” | “Symphony” evokes harmony and complexity, like a well‑balanced spice blend. |
| 5 | **Madhur Mingle** | “Sweet conversations, savory stories.” | “Madhur” means sweet in Hindi; the name invites 

#### **Conversation Buffer Memory**

**we can attach a previous mempry to remember the previous information and conversation**



```
This code implements 'Conversational Memory' using LangChain's modern LCEL approach.

1.Prompt with History: The ChatPromptTemplate includes a MessagesPlaceholder(variable_name='chat_history'). This tells the AI to look at previous messages before answering the new question.

2.Message Store: It creates a store dictionary and a function get_session_history. This keeps track of different conversations based on a unique session_id so that multiple users could have separate chats simultaneously.

3.RunnableWithMessageHistory: This is a wrapper that automatically handles the memory logic. It fetches the history from the store, injects it into the prompt, and saves the new AI response back to the history after execution.

4.Execution: By calling invoke with a session_id, the model generates a name for an 'Indian' restaurant. Because memory is active, the final print statement shows that the conversation history (both the human query and AI response) is now stored in store['session1'].

```



In [51]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

store = {}
def get_session_history(session_id: str):
    if session_id not in store: store[session_id] = ChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "Suggest a name for a {cuisine} restaurant.")
])

chain_with_memory = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="cuisine",
    history_messages_key="chat_history"
)

response = chain_with_memory.invoke(
    {"cuisine": "Indian"},
    config={"configurable": {"session_id": "session1"}}
)
print(response.content)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Below are a handful of fresh, memorable names you might consider for your Indian restaurant—each paired with a quick note on the vibe, cuisine focus, and why it could work. Feel free to mix, match, or tweak any of them to fit your concept!

| # | Restaurant Name | Vibe / Focus | Why It Works |
|---|-----------------|--------------|--------------|
| 1 | **Spice Symphony** | Upscale, fine‑dining | Evokes a musical, harmonious experience—perfect for a menu that balances classic and contemporary Indian flavors. |
| 2 | **Taj Tadka** | Classic, heritage | “Taj” nods to royalty; “Tadka” (tempering) highlights the heart‑of‑Indian cooking technique. |
| 3 | **Curry Canvas** | Artistic, modern | Suggests a creative playground of flavors—great for a place that offers signature blends or fusion dishes. |
| 4 | **Naan & Nectar** | Casual, family‑friendly | Combines the beloved flatbread with the sweetness of Indian sweets—simple, catchy, and instantly recognizable. |
| 5 | **Saffron & Silk** | Ele

#### **CONVERSATION CHAIN**

conversation buffer memory goes growing endlessly

In [26]:
!pip install -U langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.17
    Uninstalling langchain-1.3.17:
      Successfully uninstalled langchain-1.3.17


In [27]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=ConversationBufferMemory()
)

conversation.predict(input="Hi there!")



/tmp/ipykernel_847/3550379690.py:7: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory=ConversationBufferMemory()
/tmp/ipykernel_847/3550379690.py:4: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi there!
AI:

> Finished chain.


'Hey there! 👋 I’m thrilled to chat with you. I’ve got a ton of facts, stories, and trivia up my virtual sleeves—everything from the history of the Eiffel Tower to the latest breakthroughs in quantum computing, and I’m always ready to dive into whatever topic sparks your curiosity. \n\nHow’s your day going? Anything on your mind—maybe a travel destination you’re dreaming about, a book you’re reading, or a random question that’s been buzzing in your head? I’m all ears (well, all text, but you get the idea)!'

In [28]:
conversation.predict(input="Hi iam Adithya")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi there!
AI: Hey there! 👋 I’m thrilled to chat with you. I’ve got a ton of facts, stories, and trivia up my virtual sleeves—everything from the history of the Eiffel Tower to the latest breakthroughs in quantum computing, and I’m always ready to dive into whatever topic sparks your curiosity. 

How’s your day going? Anything on your mind—maybe a travel destination you’re dreaming about, a book you’re reading, or a random question that’s been buzzing in your head? I’m all ears (well, all text, but you get the idea)!
Human: Hi iam Adithya
AI:

> Finished chain.


'Hey there, Adithya! 👋 It’s great to meet you. How’s everything going on your end? Are you planning a trip, diving into a new book, or maybe just curious about something random—like the secret history of the Taj Mahal or the latest trends in AI art? Whatever it is, I’m all ears (well, all text, but you get the idea). Let me know what’s on your mind, and we can explore it together!'

### How the ConversationChain Works

When you call `conversation.predict(input="Hi there!")`, the following sequence occurs:

```text
USER INPUT: "Hi there!"
      |
      v
+-------------------------------------------------------------+
|                     ConversationChain                       |
|                                                             |
|  1. FETCH HISTORY:                                          |
|     Checks ConversationBufferMemory for past messages.      |
|     (Empty for the first message)                           |
|             |                                               |
|             v                                               |
|  2. FORMAT PROMPT:                                          |
|     Combines [System Prefix] + [History] + [User Input]     |
|             |                                               |
|             v                                               |
|  3. LLM CALL (ChatNVIDIA):                                  |
|     Sends the full formatted string to the AI model.        |
|             |                                               |
|             v                                               |
|  4. UPDATE MEMORY:                                          |
|     Saves the Human message and the AI response back        |
|     into the Buffer for the next turn.                      |
|                                                             |
+-------------------------------------------------------------+
      |
      v
AI RESPONSE: "Hey there! ..."
```

**Key Components:**
*   **`ConversationBufferMemory`**: A simple storage that keeps a list of all messages in the conversation.
*   **`verbose=True`**: This is why you saw the 'Prompt after formatting' block in your output—it shows you exactly what the `ConversationChain` sent to the model after injecting the history.

## Section 5: Document Loaders

Why it is used: Document loaders ingest unstructured file formats and convert them into a structured Document schema containing both text and metadata.

Key Parameters & Why They Are Used:
- `file_path`: Specifies the local directory path for ingestion.

Logic/Architecture Diagram:
```text
[Binary PDF] ❖ [PyPDFLoader] ❖ [Document Objects (Text + Metadata)]
```

Code Implementation:

In [29]:
!pip install pypdf

In [52]:
from langchain_classic.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/MachineLearning.ipynb - Colab.pdf")
pdf_content = loader.load()

print(f"Total Pages Loaded: {len(pdf_content)}")
print(f"Preview: {pdf_content[0].page_content[:200]}")

Total Pages Loaded: 41
Preview: MACHINE LEARNING
The train_test_split  function from sklearn.model_selection  is a
fundamental tool in machine learning used to evaluate the performance of a model.
Why use it?
When building a model, 




---



**ADITHYA UBALE**